In [8]:
"""
Authors: Isabel Godoy
Last updated: 7/22/2025

This file calculates theoretical witness expectation values for states that can be created by mixing
discrete pure states. Future state sweep files will hopefully instead sweep random computationally generated
states over a continuous range.
"""

'\nAuthors: Isabel Godoy\nLast updated: 7/22/2025\n\nThis file calculates theoretical witness expectation values for states that can be created by mixing\ndiscrete pure states. Future state sweep files will hopefully instead sweep random computationally generated\nstates over a continuous range.\n'

In [9]:
import numpy as np
import states_and_witnesses as sw
import operations as op
import process_expt as pe
import pandas as pd

initializing...


In [10]:
def get_pure_rho(state, eta, chi):
    '''
    Calculates the density matrix (rho) for a particular state with a given chi.
    
    Parameters:
        state (string): Which state we want
        chi (float): The parameter chi
    
    Returns:
        numpy.ndarray: The density matrix (rho)
    '''
    # Define kets and bell states in vector form 
    H = op.ket([1,0])
    V = op.ket([0,1])
    R = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1j)])
    L = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1j)])
    D = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (1)])
    A = op.ket([1/np.sqrt(2) * 1, 1/np.sqrt(2) * (-1)])
    
    PHI_PLUS = (np.kron(H,H) + np.kron(V,V))/np.sqrt(2)
    PHI_MINUS = (np.kron(H,H) - np.kron(V,V))/np.sqrt(2)
    PSI_PLUS = (np.kron(H,V) + np.kron(V,H))/np.sqrt(2)
    PSI_MINUS = (np.kron(H,V) - np.kron(V,H))/np.sqrt(2)
    
    ##  The following 2 states inspired the W3s
    
    if state == 'phi plus, phi minus':
        phi = np.cos(eta)*PHI_PLUS + np.exp(1j*chi)*np.sin(eta)*PHI_MINUS
    
    if state == 'psi plus, psi minus':
        phi = np.cos(eta)*PSI_PLUS + np.exp(1j*chi)*np.sin(eta)*PSI_MINUS
    
    ## The following 6 states inspired the W5s
    
    if state == 'phi plus, psi minus':
        phi = np.cos(eta)*PHI_PLUS + np.exp(1j*chi)*np.sin(eta)*PSI_MINUS
    
    if state == 'phi minus, psi plus':
        phi = np.cos(eta)*PHI_MINUS + np.exp(1j*chi)*np.sin(eta)*PSI_PLUS
    
    if state == 'phi plus, i psi plus':
        phi = np.cos(eta)*PHI_PLUS + 1j*np.exp(1j*chi)*np.sin(eta)*PSI_PLUS
    
    if state == 'phi plus, i phi minus':
        phi = np.cos(eta)*PHI_PLUS + 1j*np.exp(1j*chi)*np.sin(eta)*PHI_MINUS

    if state == 'psi plus, i psi minus':
        phi = np.cos(eta)*PSI_PLUS + 1j*np.exp(1j*chi)*np.sin(eta)*PSI_MINUS
    
    if state == 'phi minus, i psi minus':
        phi = np.cos(eta)*PHI_MINUS + 1j*np.exp(1j*chi)*np.sin(eta)*PSI_MINUS
    
    # create rho and return it
    rho = phi @ phi.conj().T

    return rho

In [11]:
def mix_rhos(state_list, state_prob, eta_chi):
    '''
    Uses above helper functions to generate a given mixed state
    
    Parameters:
    state_list (list): list of state names that are to be mixed, must match creatable state names above
    state_prob (list): probability of each state being mixed in state_list, must match index
    eta_chi (list): what eta and chi to use for each state, must match index
    
    Returns:
    rho: an NxN density matrix that results from mixing states (mixed and pure)
    '''
    
    # get individual rho's per state in state_list, taking probability into account
    individual_rhos = []
    for i, state in enumerate(state_list):
        individual_rhos.append(state_prob[i] * get_pure_rho(state, *eta_chi))

    # sum all matrices in individual rhos
    rho = np.sum(individual_rhos, axis = 0)
    
    return rho

In [12]:
def parse_W_ls(W_params, W_vals):
    """
    A function to parse the lists of outputs from witness minimization.
    Parameters:
        W_params: a list of the parameters used to minimize each witness.
        W_vals: a list of the minimum expectation value of each witness.
    Returns:
        w3_data, w8_data, w5_data: lists containing the name, value, and params of the best witness
        from each class.
    
    NOTE: this function assumes the input data is theoretical, i.e. doesn't store uncertainties.
    """

    W_names = []
    for i in range(1, 7):
        W_names.append(f'W3_{i}')
    for i in range(1, 10):
        W_names.append(f'W5_{i}')
    for i in range(1, 37):
        W_names.append(f'W8_{i}')

    # Map the names of all witnesses to their minimization params
    W_params_dict = dict(zip(W_names, W_params))

    ########
    ## W3s
    ########
    # Map the names of the W3s to their minimum expectation values
    W3_vals_dict = dict(zip(W_names[:6], W_vals[:6]))
    W3_min_name = min(W3_vals_dict, key=W3_vals_dict.get)

    # Search the dictionary for the minimum W3 and save its name,
    # expec. value, and minimization param
    w3_data = [W3_min_name, W3_vals_dict[W3_min_name], W_params_dict[W3_min_name]]

    ########
    ## W5s
    ########
    W5_vals_dict = dict(zip(W_names[6:15], W_vals[6:15]))
    W5_min_name = min(W5_vals_dict, key=W5_vals_dict.get)

    w5_data = [W5_min_name, W5_vals_dict[W5_min_name], W_params_dict[W5_min_name]]

    ########
    ## W8s
    ########
    W8_vals_dict = dict(zip(W_names[15:51], W_vals[15:51]))
    W8_min_name = min(W8_vals_dict, key=W8_vals_dict.get)

    w8_data = [W8_min_name, W8_vals_dict[W8_min_name], W_params_dict[W8_min_name]]

    return w3_data, w5_data, w8_data

In [19]:
#  Instantiate all the things we need
list_of_pure_states = ['phi plus, phi minus', 'psi plus, psi minus', 'phi plus, psi minus', 
                            'phi minus, psi plus', 'phi plus, i psi plus', 'phi plus, i phi minus',
                            'psi plus, i psi minus', 'phi minus, i psi minus']
pure_state_combos = []
for i, state_1 in enumerate(list_of_pure_states):
    for state_2 in list_of_pure_states[i:]:
        pure_state_combos.append([state_1, state_2])

etas = [np.pi/12, np.pi/6, np.pi/4, np.pi/3]
chis = np.linspace(0.001, np.pi/2, 6)
probs = [[0.5, 0.5], [0.4, 0.6], [0.3, 0.7], [0.2, 0.8], [0.1, 0.9]]

# Instantiate states to sweep over for every mixed state
deg_angles = []
rad_angles = []
for eta in etas:
    for chi in chis:
        rad_angles.append((eta, chi))

print("Total number of states to be swept:", len(pure_state_combos) * len(probs) * len(rad_angles))

Total number of states to be swept: 4320


In [20]:
"""
The big state sweep loop
"""
# Add states to this dataframe if they are witnessed at all by the W8s
not_witnessed = pd.DataFrame()
# Add states to this dataframe whenever they are witnessed by the W8s, but not the W3s or W5s
W5p_W8n = pd.DataFrame()
# Add states to this dataframe if they show really conclusive witnessing by the W8s and really poor
# witnessing by the W3s and W5s
W8_special = pd.DataFrame()

for state_pair in pure_state_combos:
    for prob in probs:
        for eta_chi in rad_angles:
            # Calculate this state's density matrix
            theo_rho = mix_rhos(state_pair, prob, eta_chi)

            # Calculate minimum expectation values for all witnesses
            W_T_params, W_T_vals = op.minimize_witnesses([sw.W3, sw.W5, sw.W8], rho=theo_rho)
            w3_data, w5_data, w8_data = parse_W_ls(W_T_params, W_T_vals)

            # Check if state was NOT witnessed well by the W3s or W5s
            if w3_data[1] > 0.01 and w5_data[1] > 0.01:
                # Save state data
                append_this = {
                    'state pair': state_pair,
                    'eta, chi': eta_chi,
                    'probabilities': prob,
                    'W3 data': w3_data,
                    'W5 data': w5_data,
                    'W8 data': w8_data
                }
                new_df_row = pd.DataFrame.from_dict([append_this])
                # Check if state was witnessed by the W8s
                if w8_data[1] < -0.01:
                    W5p_W8n = pd.concat([W5p_W8n, new_df_row])
                    print('A pretty good state was:', state_pair, eta_chi, prob)
                # Also save states that weren't witnessed at all
                elif w3_data[1] >= 0 and w5_data[1] >= 0 and w8_data[1] >= 0:
                    not_witnessed = pd.concat([not_witnessed, new_df_row])
                    print('The following state was not witnessed:', state_pair, eta_chi, prob)

            # Check if the W3s and W5s did an especially BAD job at witnessing
            if w3_data[1] > 0.2 and w5_data[1] > 0.2:
                # Check if the W8s did an especially GOOD job at witnessing
                if w8_data[1] < -0.2:
                    append_this = {
                        'state pair': state_pair,
                        'eta, chi': eta_chi,
                        'probabilities': prob,
                        'W3 data': w3_data,
                        'W5 data': w5_data,
                        'W8 data': w8_data
                    }
                    new_df_row = pd.DataFrame.from_dict([append_this])
                    W8_special = pd.concat([W8_special, new_df_row])
                    print('An amazing state was:', state_pair, eta_chi, prob)

# save the dataframes of best states to csvs
not_witnessed.to_csv('not_witnessed_by_W8_states.csv', index=False)
W5p_W8n.to_csv('W5_pos_W8_neg_states.csv', index=False)
W8_special.to_csv('W8_special_states.csv', index=False)

KeyboardInterrupt: 